# 函數呼叫 (Function Calling)

本教學使用**函數呼叫**來說明工具的使用方式。

## 什麼是函數呼叫?

**函數呼叫**是一種讓 AI 模型能夠:
1. 識別何時需要使用特定功能
2. 提取所需的參數
3. 請求執行該功能
4. 使用結果來完成任務

### 實際例子:
- 用戶問:「台北現在幾度?」
- AI 識別需要「查天氣」功能
- AI 提取參數:城市=台北
- 系統執行查天氣函數
- AI 用結果回答用戶

### 參考資源
OpenAI 的官方文件:[Function Calling 文件](https://platform.openai.com/docs/guides/function-calling?api-mode=responses)

## 環境設定

首先載入環境變數(包含 API 金鑰等敏感資訊)

In [ ]:
# 載入 .env 檔案中的環境變數
%load_ext dotenv
%dotenv ../../05_src/.secrets

In [ ]:
# 匯入必要的套件
from openai import OpenAI  # OpenAI 的 Python SDK
import json  # 用於處理 JSON 格式資料

# 建立 OpenAI 客戶端
client = OpenAI()

# 工具呼叫流程 (Tool Calling Flow)

## 核心概念

使用工具的基本想法是:**模型會主動請求使用工具**

當 AI 模型分析使用者的提示(prompt)時,它可能會判斷需要使用某個工具來完成任務。這時它會產生一種特殊的輸出,我們的程式會捕捉這個輸出並執行對應的工具。

## 完整流程圖解

```
使用者 → AI 模型 → 請求工具 → 執行工具 → AI 模型 → 最終回答
```

## 五個步驟詳解:

### 1️⃣ 發送請求給模型,並告知可用的工具
- 我們提供使用者的問題
- 同時提供工具清單(模型可以選擇使用)

### 2️⃣ 接收模型的工具呼叫請求
- 模型分析問題後決定需要哪個工具
- 回傳:要呼叫哪個函數、需要什麼參數

### 3️⃣ 在應用程式端執行程式碼
- 我們的程式實際執行該工具
- 使用模型提供的參數
- 獲得執行結果

### 4️⃣ 將工具執行結果發送回模型
- 把工具的輸出包裝成訊息
- 加入對話歷史
- 再次呼叫模型

### 5️⃣ 接收最終回應(或更多工具呼叫)
- 模型用工具結果生成最終答案
- 或者,模型可能需要呼叫更多工具

## 重要觀念

- **雙向溝通**:我們需要與 AI 模型進行至少兩次對話
- **中間執行**:在兩次對話之間,我們在本地執行實際功能
- **上下文保持**:需要記錄整個對話歷史

![工具呼叫流程圖](./img/05_function-calling-diagram-steps.png)

# 步驟 1: 定義可用的工具

## 工具定義結構

工具定義包含:
- **函數名稱** - 工具的識別名稱
- **參數** - 工具需要什麼輸入
- **描述** - 幫助 AI 理解何時該用這個工具
- **其他元資料** - 額外的配置資訊

## 本例:星座運勢工具

In [ ]:
# 定義工具清單
tools = [
    {
        "type": "function",  # 類型:函數
        "name": "get_horoscope",  # 函數名稱
        "description": "取得指定星座的運勢預測。",  # 描述:幫助 AI 理解這個工具的用途
        "parameters": {  # 參數定義
            "type": "object",  # 參數是一個物件
            "properties": {  # 物件的屬性
                "zodiac_sign": {  # 參數名稱:星座
                    "type": "string",  # 資料類型:字串
                    "description": "星座名稱,例如:白羊座(Aries)、金牛座(Taurus)、水瓶座(Aquarius)",  # 參數說明
                }
            },
            "required": ["zodiac_sign"],  # 必填參數
            "additionalProperties": False,  # 不允許額外的屬性
        },
        "strict": True,  # 嚴格模式:確保回傳格式正確
    },
]

# 實際執行的函數
def get_horoscope(zodiac_sign: str) -> str:
    """
    取得星座運勢的函數
    
    參數:
        zodiac_sign (str): 星座名稱
    
    回傳:
        str: 運勢文字
    
    註: 這是示範用的簡化版本,實際應用中可能會:
    - 連接真實的星座運勢 API
    - 從資料庫查詢
    - 使用更複雜的生成邏輯
    """
    # 示範用的固定回應
    horoscope = f"{zodiac_sign}: 今天是開啟新事物的好日子。"
    return horoscope

# 補充說明:十二星座對照
# 白羊座 - Aries
# 金牛座 - Taurus  
# 雙子座 - Gemini
# 巨蟹座 - Cancer
# 獅子座 - Leo
# 處女座 - Virgo
# 天秤座 - Libra
# 天蠍座 - Scorpio
# 射手座 - Sagittarius
# 摩羯座 - Capricorn
# 水瓶座 - Aquarius
# 雙魚座 - Pisces

## 為什麼需要「對話記憶」?

在使用工具時,我們需要保留對話的「記憶」。

### 類比理解:
想像你在打電話:
1. 你問朋友:「我是水瓶座,今天運勢如何?」
2. 朋友說:「等我查一下運勢書...」(這是工具呼叫)
3. 朋友查完後告訴你結果

如果朋友在步驟 3 忘記你問的是什麼,對話就無法繼續!

### 技術實現:
我們使用 `input_list` 來儲存:
- 使用者的原始問題
- 模型的工具呼叫請求  
- 工具執行的結果
- 所有相關的上下文

In [ ]:
# 建立對話歷史清單
input_list = [
    {
        "role": "user",  # 角色:使用者
        "content": "我的星座運勢如何?我是水瓶座。"  # 使用者的問題
    }
]

# 這個清單會在整個流程中不斷增加新的訊息

## 第一次呼叫模型

我們將使用者的問題和可用工具一起送給 AI 模型

In [ ]:
# 發送請求給 OpenAI API
response = client.responses.create(
    model="gpt-5",  # 使用的模型
    tools=tools,  # 提供可用的工具清單
    input=input_list,  # 輸入對話歷史
)

# 這時 AI 會分析:
# 1. 使用者問的是星座運勢
# 2. 我有 get_horoscope 這個工具可以用
# 3. 使用者說他是水瓶座 (Aquarius)
# 4. 我應該呼叫 get_horoscope(zodiac_sign='Aquarius')

# 步驟 3: 在應用程式端執行工具

收到模型的回應後,我們要在本地執行實際的功能

## 檢查模型的輸出

模型的回應中包含:
1. **推理項目** (reasoning item) - AI 的思考過程
2. **函數工具呼叫** (function tool call) - AI 請求執行的工具

In [ ]:
# 查看完整的輸出
response.output

# 你會看到類似這樣的結構:
# [
#   <reasoning_item>,  # AI 的推理過程
#   <function_call_item>  # 工具呼叫請求
# ]

## 詳細查看函數呼叫

函數呼叫項目包含模型希望執行的具體資訊

In [ ]:
# 印出第二個項目(函數呼叫)的詳細資訊
print(response.output[1].to_json())

# 輸出會顯示:
# {
#   "type": "function_call",
#   "name": "get_horoscope",
#   "arguments": "{\"zodiac_sign\": \"Aquarius\"}",
#   "call_id": "call_abc123"  # 用於追蹤的唯一 ID
# }

# 這表示:模型請求執行 get_horoscope(zodiac_sign='Aquarius')

## 執行工具並記錄結果

這是關鍵步驟!我們要:
1. 執行 `get_horoscope` 函數
2. 將結果加入對話歷史

In [ ]:
# 首先,將模型的輸出加入對話歷史
input_list += response.output

# 遍歷模型輸出的每個項目
for item in response.output:
    # 檢查是否為函數呼叫
    if item.type == "function_call":
        # 檢查是否為 get_horoscope 函數
        if item.name == "get_horoscope":
            # 執行函數邏輯
            # **json.loads(item.arguments) 會將 JSON 字串轉成 Python 字典
            # ** 運算子會將字典展開成關鍵字參數
            # 例如: {"zodiac_sign": "Aquarius"} 變成 zodiac_sign="Aquarius"
            horoscope = get_horoscope(**json.loads(item.arguments))
            
            # 將函數執行結果加入對話歷史
            input_list.append({
                "type": "function_call_output",  # 類型:函數輸出
                "call_id": item.call_id,  # 對應的呼叫 ID
                "output": json.dumps({  # 輸出結果(轉成 JSON 格式)
                    "horoscope": horoscope
                })
            })

# 現在 input_list 包含:
# 1. 使用者的原始問題
# 2. AI 的推理過程
# 3. AI 的函數呼叫請求
# 4. 函數執行的結果

# 步驟 4: 將工具結果發送回模型

現在我們要進行第二次模型呼叫,提供工具執行的結果

## 檢查完整的對話歷史

在第二次呼叫前,先看看我們準備傳送給模型的完整上下文

In [ ]:
# 印出完整的 input_list
print(input_list)

# 你會看到完整的對話流程:
# [
#   {使用者問題},
#   {AI 推理},
#   {AI 函數呼叫請求},
#   {函數執行結果}
# ]

## 第二次呼叫模型

In [ ]:
# 發送第二次請求
response = client.responses.create(
    model="gpt-5",
    instructions="只回應由工具生成的星座運勢。",  # 明確指示 AI 如何使用工具結果
    tools=tools,  # 仍然提供工具清單(以防需要更多工具呼叫)
    input=input_list,  # 包含完整對話歷史的輸入
)

# 這次 AI 會:
# 1. 看到之前它請求的工具已經執行
# 2. 讀取工具的執行結果
# 3. 用這個結果來回答使用者的原始問題

# 步驟 5: 接收最終回應

模型現在有了工具執行的結果,可以給出完整的答案了!

In [ ]:
# 印出完整的回應物件(JSON 格式)
print(response.model_dump_json(indent=2))

# 印出最終的文字回應
print("\n" + response.output_text)

# 最終回應會是類似:
# "Aquarius: 今天是開啟新事物的好日子。"

---

# 總結與重點回顧

## 🎯 核心概念

1. **工具擴展 AI 能力** - 讓 AI 能做更多事情
2. **雙向對話流程** - 至少需要兩次 API 呼叫
3. **中間執行邏輯** - 在本地執行實際功能
4. **保持上下文** - 記錄完整對話歷史

## 📋 完整流程回顧

```
1. 使用者提問 + 提供工具清單
   ↓
2. AI 分析並請求工具
   ↓
3. 本地執行工具函數
   ↓
4. 將結果回傳給 AI
   ↓
5. AI 生成最終答案
```

## 💡 實際應用場景

- **客服機器人** - 查詢訂單狀態、退款處理
- **個人助理** - 查天氣、設定提醒、查詢行事曆
- **資料分析** - 查詢資料庫、生成圖表
- **系統控制** - 控制智慧家居、執行系統指令

## 🚀 進階學習方向

1. **多工具協作** - 一次對話中使用多個工具
2. **工具鏈** - 一個工具的輸出作為另一個工具的輸入
3. **錯誤處理** - 當工具執行失敗時該如何處理
4. **安全性** - 如何限制工具的存取權限
5. **效能優化** - 如何減少 API 呼叫次數

## ⚠️ 注意事項

- **成本控制** - 每次工具呼叫都需要多次 API 呼叫
- **回應時間** - 工具執行會增加整體回應時間
- **錯誤處理** - 需要處理工具執行失敗的情況
- **安全性** - 小心驗證工具的輸入參數

---

## 📚 延伸閱讀

- [OpenAI Function Calling 官方文件](https://platform.openai.com/docs/guides/function-calling)
- [Model Context Protocol (MCP) 介紹](https://modelcontextprotocol.io/)
- [LangChain Tools 文件](https://python.langchain.com/docs/modules/agents/tools/)

---

**恭喜你完成這個教學!** 🎉

現在你已經理解如何讓 AI 模型使用工具來擴展它的能力。試著修改程式碼,創建你自己的工具吧!